In [1]:
import pandas as pd
import numpy as np
import joblib


In [2]:
# Load the same movie_data used in the recommender
movie_data = pd.read_csv("../../datasets/processed/movie_data.csv")
movie_data = movie_data.reset_index(drop=True)
movie_data["content"] = movie_data["content"].fillna("")
movie_data = movie_data.drop_duplicates(subset="movieId")
movie_data = movie_data.reset_index(drop=True)


ratings = pd.read_csv("../../datasets/raw/movielens/ratings.csv")

print("movie_data shape:", movie_data.shape)
print("ratings shape:", ratings.shape)
ratings.head()

movie_data shape: (3650, 11)
ratings shape: (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
# Define "relevant" as movies a user rated 4.0 or higher
RELEVANCE_THRESHOLD = 4.0

relevant_ratings = ratings[ratings["rating"] >= RELEVANCE_THRESHOLD]

print("Total ratings:", len(ratings))
print("Relevant ratings (>= 4.0):", len(relevant_ratings))

# Pick a sample user to test with
sample_user_id = relevant_ratings["userId"].value_counts().index[0]
print("\nSample user for testing:", sample_user_id)

user_relevant_movies = relevant_ratings[
    relevant_ratings["userId"] == sample_user_id
]["movieId"].tolist()

print("Number of relevant movies for this user:", len(user_relevant_movies))

Total ratings: 100836
Relevant ratings (>= 4.0): 48580

Sample user for testing: 414
Number of relevant movies for this user: 1227


In [4]:
# Build a movieId -> clean_title mapping using movie_data
movieid_to_title = pd.Series(
    movie_data["clean_title"].values,
    index=movie_data["movieId"]
).to_dict()

# Keep only the relevant movies that also exist in our movie_data
# (some movies might have been dropped during preprocessing)
user_relevant_titles = [
    movieid_to_title[m] for m in user_relevant_movies if m in movieid_to_title
]

print("User's relevant movies found in movie_data:", len(user_relevant_titles))
print(user_relevant_titles[:10])

User's relevant movies found in movie_data: 1061
['Toy Story', 'Grumpier Old Men', 'American President, The', 'Sense and Sensibility', 'Get Shorty', 'Twelve Monkeys (a.k.a. 12 Monkeys)', 'Babe', 'Clueless', 'Seven (a.k.a. Se7en)', 'Usual Suspects, The']


In [5]:
import sys
sys.path.append("..")  # so we can import from ai-service root

from recommendation.tfidf_recommender import recommend_movies


def precision_recall_at_k(user_relevant_titles, k=10, max_queries=50):
    """
    For each relevant movie (used as a query), check how many of the
    OTHER relevant movies for that user appear in the top-K recommendations.
    """
    precisions = []
    recalls = []

    relevant_set = set(user_relevant_titles)

    # Limit how many queries we run (for speed)
    query_titles = user_relevant_titles[:max_queries]

    for query_title in query_titles:
        result = recommend_movies(query_title, top_n=k)

        if isinstance(result, str):
            continue  # "Movie not found."

        recommended_titles = set(result["clean_title"].tolist())

        # The "ground truth" is all OTHER relevant movies (excluding the query itself)
        ground_truth = relevant_set - {query_title}

        hits = recommended_titles & ground_truth

        precision = len(hits) / k
        recall = len(hits) / len(ground_truth) if ground_truth else 0

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls), len(precisions)


avg_precision, avg_recall, num_queries = precision_recall_at_k(
    user_relevant_titles, k=10, max_queries=50
)

print(f"Evaluated on {num_queries} queries")
print(f"Precision@10: {avg_precision:.4f}")
print(f"Recall@10: {avg_recall:.4f}")

Evaluated on 50 queries
Precision@10: 0.2160
Recall@10: 0.0020


In [6]:
def evaluate_multiple_users(num_users=10, k=10, max_queries_per_user=20):
    """
    Run precision/recall evaluation across multiple users and average the results.
    """
    # Pick users with enough relevant ratings (at least 5) for meaningful evaluation
    user_counts = relevant_ratings["userId"].value_counts()
    eligible_users = user_counts[user_counts >= 5].index.tolist()

    sample_users = eligible_users[:num_users]

    all_precisions = []
    all_recalls = []

    for uid in sample_users:
        movies_for_user = relevant_ratings[
            relevant_ratings["userId"] == uid
        ]["movieId"].tolist()

        titles_for_user = [
            movieid_to_title[m] for m in movies_for_user if m in movieid_to_title
        ]

        if len(titles_for_user) < 2:
            continue

        p, r, n = precision_recall_at_k(
            titles_for_user, k=k, max_queries=max_queries_per_user
        )

        if n > 0:
            all_precisions.append(p)
            all_recalls.append(r)

    print(f"Evaluated across {len(all_precisions)} users")
    print(f"Average Precision@{k}: {np.mean(all_precisions):.4f}")
    print(f"Average Recall@{k}: {np.mean(all_recalls):.4f}")

    return np.mean(all_precisions), np.mean(all_recalls)


baseline_precision, baseline_recall = evaluate_multiple_users(
    num_users=10, k=10, max_queries_per_user=20
)

Evaluated across 10 users
Average Precision@10: 0.1695
Average Recall@10: 0.0033


In [8]:
def f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


baseline_f1 = f1_score(baseline_precision, baseline_recall)

print(f"Precision@10: {baseline_precision:.4f}")
print(f"Recall@10:    {baseline_recall:.4f}")
print(f"F1 Score@10:  {baseline_f1:.4f}")

# Update the saved results file to include F1
evaluation_results["f1_score_at_10"] = round(float(baseline_f1), 4)

with open("../evaluation/baseline_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=4)

print("\nUpdated baseline_results.json with F1 score")
print(json.dumps(evaluation_results, indent=4))

Precision@10: 0.1695
Recall@10:    0.0033
F1 Score@10:  0.0065

Updated baseline_results.json with F1 score
{
    "model": "TF-IDF + Cosine Similarity (Content-Based, Baseline)",
    "k": 10,
    "num_users_evaluated": 10,
    "precision_at_10": 0.1695,
    "recall_at_10": 0.0033,
    "relevance_threshold": 4.0,
    "f1_score_at_10": 0.0065
}


In [7]:
import json
import os

evaluation_results = {
    "model": "TF-IDF + Cosine Similarity (Content-Based, Baseline)",
    "k": 10,
    "num_users_evaluated": 10,
    "precision_at_10": round(float(baseline_precision), 4),
    "recall_at_10": round(float(baseline_recall), 4),
    "relevance_threshold": RELEVANCE_THRESHOLD
}

os.makedirs("../evaluation", exist_ok=True)

with open("../evaluation/baseline_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=4)

print("Baseline results saved to ai-service/evaluation/baseline_results.json")
print(json.dumps(evaluation_results, indent=4))

Baseline results saved to ai-service/evaluation/baseline_results.json
{
    "model": "TF-IDF + Cosine Similarity (Content-Based, Baseline)",
    "k": 10,
    "num_users_evaluated": 10,
    "precision_at_10": 0.1695,
    "recall_at_10": 0.0033,
    "relevance_threshold": 4.0
}
